## Dream 2000 offers scraping

In [1]:
import selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1500,950")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- offers List --
offers = []


## -- Scraping --
driver = webdriver.Chrome(options=options)
driver.get("https://dream2000.com/en/collections/wearables-offers")
time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(By.CSS_SELECTOR,'.product-card__wrapper.color-scheme-theme.anm-reveal-item.overflow-hidden')

    for card in cards:

        try:
            Product = card.find_element(By.CSS_SELECTOR, 'a[href*="/en/products/"]')

            Name = Product.get_attribute('aria-label')
            Link = Product.get_attribute('href')
            Image = card.find_element(By.CSS_SELECTOR, 'img.product-card__image').get_attribute('src')

        except NoSuchElementException:
            continue

        ## -- Price --
        try:
            Price = card.find_element(By.CSS_SELECTOR, '.price').text

        except NoSuchElementException:
            Price = "Product Not Available"

        ## -- Stock --
        try:
            Stock = card.find_element(By.CSS_SELECTOR,'[data-inventory-status]').get_attribute('data-inventory-status')

        except NoSuchElementException:
            Stock = "Product Not Available"

        offers.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Price': Price,
            'Stock': Stock
        })

        page_products += 1

    ## -- Page Summary --

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(offers)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute('href')

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(offers)}")


Page 1 finished successfully. Products collected on this page: 18. Total products collected: 18
No Next page. Scraping finished.

Scraping completed. Total products: 18


In [7]:
offers

[{'Name': 'Mi Redmi Buds 3T Pro',
  'Link': 'https://dream2000.com/en/products/mi-redmi-buds-3t-pro',
  'Image': 'https://dream2000.com/cdn/shop/files/mi-redmi-buds-3t-pro---white_1.jpg?v=1779579470',
  'Price': 'LE 1,711.00\nLE 1,999.00',
  'Stock': 'in-stock'},
 {'Name': 'itel Smart Watch Horizon ISW-N8',
  'Link': 'https://dream2000.com/en/products/itel-smart-watch-horizon-isw-n8-meteorite',
  'Image': 'https://dream2000.com/cdn/shop/files/new_project_-_2024-11-14t134630.542.jpg?v=1779592565',
  'Price': 'LE 1,611.00\nLE 1,899.00',
  'Stock': 'low-stock'},
 {'Name': 'Itel Unicorn Smart Watch ISW-N9',
  'Link': 'https://dream2000.com/en/products/itel-unicorn-smart-watch-dark-chrome',
  'Image': 'https://dream2000.com/cdn/shop/files/2024-11-24t095054.345.jpg?v=1779592924',
  'Price': 'LE 1,411.00\nLE 1,699.00',
  'Stock': 'in-stock'},
 {'Name': 'Xiaomi Redmi Buds 6 Pro',
  'Link': 'https://dream2000.com/en/products/redmi-buds-6-pro',
  'Image': 'https://dream2000.com/cdn/shop/files/20

## Import To CSV & Parquet File


In [5]:
import pandas as pd

In [7]:
df_offers = pd.DataFrame(offers)
df_offers.head()

,Name,Link,Image,Price,Stock
0,Mi Redmi Buds 3T Pro,https://dream2000.com/en/products/mi-redmi-bud...,https://dream2000.com/cdn/shop/files/mi-redmi-...,"LE 1,711.00\nLE 1,999.00",in-stock
1,itel Smart Watch Horizon ISW-N8,https://dream2000.com/en/products/itel-smart-w...,https://dream2000.com/cdn/shop/files/new_proje...,"LE 1,611.00\nLE 1,899.00",low-stock
2,Itel Unicorn Smart Watch ISW-N9,https://dream2000.com/en/products/itel-unicorn...,https://dream2000.com/cdn/shop/files/2024-11-2...,"LE 1,411.00\nLE 1,699.00",in-stock
3,Xiaomi Redmi Buds 6 Pro,https://dream2000.com/en/products/redmi-buds-6...,https://dream2000.com/cdn/shop/files/2025-02-2...,"LE 3,911.00\nLE 4,700.00",in-stock
4,Recci Freepods Amber REP-W50,https://dream2000.com/en/products/recci-freepo...,https://dream2000.com/cdn/shop/files/recci-fre...,LE 411.00\nLE 499.00,in-stock


In [8]:
df_offers.to_csv('dream2000_offers.csv', index=False)

In [9]:
df_offers.to_parquet('dream2000_offers.parquet', index=False)